# Demo 1 — Document Ingestion & Embedding

This notebook loads the synthetic tax policy documents, chunks them, embeds them using Ollama (`nomic-embed-text`), and stores them in ChromaDB.

**Prerequisites (running locally against Docker services):**
```
cd demo1-phoenix-rag
docker compose up -d phoenix chromadb ollama
# wait for model-puller to finish, then:
pip install -r requirements-dev.txt
```

Phoenix UI: http://localhost:6006  
ChromaDB API: http://localhost:8000

In [1]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")

# When running the notebook locally (not in Docker), services are on localhost
OLLAMA_BASE_URL = "http://localhost:11434"
PHOENIX_COLLECTOR_ENDPOINT = "http://localhost:6006"
CHROMA_HOST = "localhost"
CHROMA_PORT = 8000
COLLECTION_NAME = "tax_docs"
DOCS_DIR = "../data/docs"

print("Environment loaded.")

Environment loaded.


In [2]:
# Register Phoenix as the OTLP trace collector
# Traces from this notebook will appear in the Phoenix UI under project 'tax-rag-ingest'
from phoenix.otel import register
from openinference.instrumentation.langchain import LangChainInstrumentor

register(
    endpoint=PHOENIX_COLLECTOR_ENDPOINT,
    project_name="tax-rag-ingest",
)
LangChainInstrumentor().instrument()

print(f"Phoenix tracing active → {PHOENIX_COLLECTOR_ENDPOINT}")

🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: tax-rag-ingest
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: http://localhost:6006
|  Transport: HTTP + protobuf
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.

Phoenix tracing active → http://localhost:6006


/Users/zaidansari/Documents/Development/ai-exploration/demo1-phoenix-rag/.venv/lib/python3.13/site-packages/phoenix/otel/otel.py:433: UserWarning: Could not infer collector endpoint protocol, defaulting to HTTP.
  warnings.warn("Could not infer collector endpoint protocol, defaulting to HTTP.")


In [3]:
# Load all .txt documents from the docs directory
from langchain_community.document_loaders import TextLoader, PyPDFLoader

docs = []
for fname in sorted(os.listdir(DOCS_DIR)):
    path = os.path.join(DOCS_DIR, fname)
    if fname.endswith(".pdf"):
        loader = PyPDFLoader(path)
    elif fname.endswith(".txt"):
        loader = TextLoader(path)
    else:
        continue
    loaded = loader.load()
    docs.extend(loaded)
    print(f"  Loaded: {fname} ({len(loaded)} doc(s))")

print(f"\nTotal documents loaded: {len(docs)}")

  Loaded: capital_gains_tax_2024.txt (1 doc(s))
  Loaded: retirement_accounts_2024.txt (1 doc(s))
  Loaded: standard_deduction_2024.txt (1 doc(s))
  Loaded: tax_brackets_2024.txt (1 doc(s))
  Loaded: w2_income_deductions_2024.txt (1 doc(s))

Total documents loaded: 5


In [4]:
# Chunk documents using RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)

print(f"Total chunks: {len(chunks)}")
print("\nSample chunk:")
print(f"  Content: {chunks[0].page_content[:200]}...")
print(f"  Metadata: {chunks[0].metadata}")

Total chunks: 34

Sample chunk:
  Content: CAPITAL GAINS TAX — TAX YEAR 2024

OVERVIEW

A capital gain is the profit realized when you sell a capital asset (stocks, bonds, real estate, collectibles) for more than its adjusted basis. Capital ga...
  Metadata: {'source': '../data/docs/capital_gains_tax_2024.txt'}


In [5]:
# Initialize Ollama embeddings (nomic-embed-text produces 768-dim vectors)
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="nomic-embed-text", base_url=OLLAMA_BASE_URL)

# Quick sanity check
test_vec = embeddings.embed_query("What is the standard deduction?")
print(f"Embedding dimensions: {len(test_vec)}")
print(f"First 5 values: {test_vec[:5]}")

Embedding dimensions: 768
First 5 values: [0.035229817, 0.049746577, -0.1717721, -0.035796586, 0.035843797]


In [6]:
# Store chunks in ChromaDB
import chromadb
from langchain_chroma import Chroma

client = chromadb.HttpClient(host=CHROMA_HOST, port=CHROMA_PORT)

# Drop and recreate collection for a clean demo run
try:
    client.delete_collection(COLLECTION_NAME)
    print(f"Deleted existing collection '{COLLECTION_NAME}'")
except Exception:
    pass

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    client=client,
    collection_name=COLLECTION_NAME,
)

count = client.get_collection(COLLECTION_NAME).count()
print(f"\nStored {count} vectors in ChromaDB collection '{COLLECTION_NAME}'")

Deleted existing collection 'tax_docs'

Stored 34 vectors in ChromaDB collection 'tax_docs'


In [7]:
# Verify retrieval works
test_queries = [
    "What is the standard deduction for a single filer?",
    "401k contribution limits",
    "long term capital gains tax rate",
]

for q in test_queries:
    results = vectorstore.similarity_search(q, k=2)
    print(f"Query: '{q}'")
    for i, doc in enumerate(results):
        src = doc.metadata.get("source", "?").split("/")[-1]
        print(f"  [{i+1}] {src}: {doc.page_content[:120]}...")
    print()

Query: 'What is the standard deduction for a single filer?'
  [1] standard_deduction_2024.txt: INTERACTION WITH AMT

High-income taxpayers who itemize may be subject to the Alternative Minimum Tax (AMT). The AMT exe...
  [2] w2_income_deductions_2024.txt: ABOVE-THE-LINE DEDUCTIONS (ADJUSTMENTS TO INCOME)

These deductions reduce your Adjusted Gross Income (AGI) and are avai...

Query: '401k contribution limits'
  [1] retirement_accounts_2024.txt: SIMPLE IRA: Employee contribution limit $16,000 ($19,500 with catch-up for age 50+). Employers must either match up to 3...
  [2] retirement_accounts_2024.txt: RETIREMENT ACCOUNT CONTRIBUTION LIMITS AND TAX RULES — TAX YEAR 2024

401(k), 403(b), AND 457(b) PLANS

Employee Contrib...

Query: 'long term capital gains tax rate'
  [1] capital_gains_tax_2024.txt: SHORT-TERM CAPITAL GAINS

Assets held for one year or less before sale are subject to short-term capital gains tax, whic...
  [2] capital_gains_tax_2024.txt: CAPITAL GAINS TAX — TAX YEAR 2

## Next Steps

Ingestion is complete. Open the Phoenix UI at **http://localhost:6006** to see the ingestion traces.

To run the full RAG query pipeline with tracing:
```bash
docker compose up rag-app
```
Or run queries directly from Python:
```python
import sys; sys.path.insert(0, '..')
from app.rag_pipeline import query_rag
result = query_rag("What is the Roth IRA income limit for 2024?")
print(result['answer'])
```